![Piggy bank](piggy_bank.jpg)

Personal loans are a lucrative revenue stream for banks. The typical interest rate of a two-year loan in the United Kingdom is [around 10%](https://www.experian.com/blogs/ask-experian/whats-a-good-interest-rate-for-a-personal-loan/). This might not sound like a lot, but in September 2022 alone UK consumers borrowed [around £1.5 billion](https://www.ukfinance.org.uk/system/files/2022-12/Household%20Finance%20Review%202022%20Q3-%20Final.pdf), which would mean approximately £300 million in interest generated by banks over two years!

You have been asked to work with a bank to clean the data they collected as part of a recent marketing campaign, which aimed to get customers to take out a personal loan. They plan to conduct more marketing campaigns going forward so would like you to ensure it conforms to the specific structure and data types that they specify so that they can then use the cleaned data you provide to set up a PostgreSQL database, which will store this campaign's data and allow data from future campaigns to be easily imported. 

They have supplied you with a csv file called `"bank_marketing.csv"`, which you will need to clean, reformat, and split the data, saving three final csv files. Specifically, the three files should have the names and contents as outlined below:

## `client.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `age` | `integer` | Client's age in years | N/A |
| `job` | `object` | Client's type of job | Change `"."` to `"_"` |
| `marital` | `object` | Client's marital status | N/A |
| `education` | `object` | Client's level of education | Change `"."` to `"_"` and `"unknown"` to `np.NaN` |
| `credit_default` | `bool` | Whether the client's credit is in default | Convert to `boolean` data type:<br> `1` if `"yes"`, otherwise `0` |
| `mortgage` | `bool` | Whether the client has an existing mortgage (housing loan) | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0` |

<br>

## `campaign.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `number_contacts` | `integer` | Number of contact attempts to the client in the current campaign | N/A |
| `contact_duration` | `integer` | Last contact duration in seconds | N/A |
| `previous_campaign_contacts` | `integer` | Number of contact attempts to the client in the previous campaign | N/A |
| `previous_outcome` | `bool` | Outcome of the previous campaign | Convert to boolean data type:<br> `1` if `"success"`, otherwise `0`. |
| `campaign_outcome` | `bool` | Outcome of the current campaign | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0`. |
| `last_contact_date` | `datetime` | Last date the client was contacted | Create from a combination of `day`, `month`, and a newly created `year` column (which should have a value of `2022`); <br> **Format =** `"YYYY-MM-DD"` |

<br>

## `economics.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `cons_price_idx` | `float` | Consumer price index (monthly indicator) | N/A |
| `euribor_three_months` | `float` | Euro Interbank Offered Rate (euribor) three-month rate (daily indicator) | N/A |

In [3]:
import pandas as pd
import numpy as np

# Start coding here...

In [4]:
df = pd.read_csv("data/bank_marketing.csv")

for col in ["credit_default", "mortgage", "previous_outcome", "campaign_outcome"]:
    print(col)
    print("--------------")
    print(df[col].value_counts())

credit_default
--------------
credit_default
no         32588
unknown     8597
yes            3
Name: count, dtype: int64
mortgage
--------------
mortgage
yes        21576
no         18622
unknown      990
Name: count, dtype: int64
previous_outcome
--------------
previous_outcome
nonexistent    35563
failure         4252
success         1373
Name: count, dtype: int64
campaign_outcome
--------------
campaign_outcome
no     36548
yes     4640
Name: count, dtype: int64


In [5]:
df.head()

,client_id,age,job,marital,education,credit_default,mortgage,month,day,contact_duration,number_contacts,previous_campaign_contacts,previous_outcome,cons_price_idx,euribor_three_months,campaign_outcome
0,0,56,housemaid,married,basic.4y,no,no,may,13,261,1,0,nonexistent,93.994,4.857,no
1,1,57,services,married,high.school,unknown,no,may,19,149,1,0,nonexistent,93.994,4.857,no
2,2,37,services,married,high.school,no,yes,may,23,226,1,0,nonexistent,93.994,4.857,no
3,3,40,admin.,married,basic.6y,no,no,may,27,151,1,0,nonexistent,93.994,4.857,no
4,4,56,services,married,high.school,no,no,may,3,307,1,0,nonexistent,93.994,4.857,no


In [6]:
df.tail()

,client_id,age,job,marital,education,credit_default,mortgage,month,day,contact_duration,number_contacts,previous_campaign_contacts,previous_outcome,cons_price_idx,euribor_three_months,campaign_outcome
41183,41183,73,retired,married,professional.course,no,yes,nov,30,334,1,0,nonexistent,94.767,1.028,yes
41184,41184,46,blue-collar,married,professional.course,no,no,nov,6,383,1,0,nonexistent,94.767,1.028,no
41185,41185,56,retired,married,university.degree,no,yes,nov,24,189,2,0,nonexistent,94.767,1.028,no
41186,41186,44,technician,married,professional.course,no,no,nov,17,442,1,0,nonexistent,94.767,1.028,yes
41187,41187,74,retired,married,professional.course,no,yes,nov,23,239,3,1,failure,94.767,1.028,no


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   client_id                   41188 non-null  int64  
 1   age                         41188 non-null  int64  
 2   job                         41188 non-null  str    
 3   marital                     41188 non-null  str    
 4   education                   41188 non-null  str    
 5   credit_default              41188 non-null  str    
 6   mortgage                    41188 non-null  str    
 7   month                       41188 non-null  str    
 8   day                         41188 non-null  int64  
 9   contact_duration            41188 non-null  int64  
 10  number_contacts             41188 non-null  int64  
 11  previous_campaign_contacts  41188 non-null  int64  
 12  previous_outcome            41188 non-null  str    
 13  cons_price_idx              41188 non-null

In [8]:
df.describe()

,client_id,age,day,contact_duration,number_contacts,previous_campaign_contacts,cons_price_idx,euribor_three_months
count,41188.00000,41188.00000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000
mean,20593.50000,40.02406,15.730334,258.285010,2.567593,0.172963,93.575664,3.621291
std,11890.09578,10.42125,8.889867,259.279249,2.770014,0.494901,0.578840,1.734447
min,0.00000,17.00000,1.000000,0.000000,1.000000,0.000000,92.201000,0.634000
25%,10296.75000,32.00000,8.000000,102.000000,1.000000,0.000000,93.075000,1.344000
50%,20593.50000,38.00000,16.000000,180.000000,2.000000,0.000000,93.749000,4.857000
75%,30890.25000,47.00000,23.000000,319.000000,3.000000,0.000000,93.994000,4.961000
max,41187.00000,98.00000,31.000000,4918.000000,56.000000,7.000000,94.767000,5.045000


In [15]:
client = df[["client_id","age","job","marital","education","credit_default","mortgage"]]
client.head()
client.info()
client.describe(include="all")

<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   client_id       41188 non-null  int64
 1   age             41188 non-null  int64
 2   job             41188 non-null  str  
 3   marital         41188 non-null  str  
 4   education       41188 non-null  str  
 5   credit_default  41188 non-null  str  
 6   mortgage        41188 non-null  str  
dtypes: int64(2), str(5)
memory usage: 2.2 MB


,client_id,age,job,marital,education,credit_default,mortgage
count,41188.00000,41188.00000,41188,41188,41188,41188,41188
unique,NaN,NaN,12,4,8,3,3
top,NaN,NaN,admin.,married,university.degree,no,yes
freq,NaN,NaN,10422,24928,12168,32588,21576
mean,20593.50000,40.02406,NaN,NaN,NaN,NaN,NaN
std,11890.09578,10.42125,NaN,NaN,NaN,NaN,NaN
min,0.00000,17.00000,NaN,NaN,NaN,NaN,NaN
25%,10296.75000,32.00000,NaN,NaN,NaN,NaN,NaN
50%,20593.50000,38.00000,NaN,NaN,NaN,NaN,NaN
75%,30890.25000,47.00000,NaN,NaN,NaN,NaN,NaN


In [18]:
campaign = df[["client_id","number_contacts","contact_duration","previous_campaign_contacts","previous_outcome","campaign_outcome"]]
campaign.head()
campaign.info()
campaign.describe(include="all")

<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 6 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   client_id                   41188 non-null  int64
 1   number_contacts             41188 non-null  int64
 2   contact_duration            41188 non-null  int64
 3   previous_campaign_contacts  41188 non-null  int64
 4   previous_outcome            41188 non-null  str  
 5   campaign_outcome            41188 non-null  str  
dtypes: int64(4), str(2)
memory usage: 1.9 MB


,client_id,number_contacts,contact_duration,previous_campaign_contacts,previous_outcome,campaign_outcome
count,41188.00000,41188.000000,41188.000000,41188.000000,41188,41188
unique,NaN,NaN,NaN,NaN,3,2
top,NaN,NaN,NaN,NaN,nonexistent,no
freq,NaN,NaN,NaN,NaN,35563,36548
mean,20593.50000,2.567593,258.285010,0.172963,NaN,NaN
std,11890.09578,2.770014,259.279249,0.494901,NaN,NaN
min,0.00000,1.000000,0.000000,0.000000,NaN,NaN
25%,10296.75000,1.000000,102.000000,0.000000,NaN,NaN
50%,20593.50000,2.000000,180.000000,0.000000,NaN,NaN
75%,30890.25000,3.000000,319.000000,0.000000,NaN,NaN


In [21]:
economics = df[["client_id","cons_price_idx","euribor_three_months"]]
economics.head()
economics.info()
economics.describe(include="all")

<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   client_id             41188 non-null  int64  
 1   cons_price_idx        41188 non-null  float64
 2   euribor_three_months  41188 non-null  float64
dtypes: float64(2), int64(1)
memory usage: 965.5 KB


,client_id,cons_price_idx,euribor_three_months
count,41188.00000,41188.000000,41188.000000
mean,20593.50000,93.575664,3.621291
std,11890.09578,0.578840,1.734447
min,0.00000,92.201000,0.634000
25%,10296.75000,93.075000,1.344000
50%,20593.50000,93.749000,4.857000
75%,30890.25000,93.994000,4.961000
max,41187.00000,94.767000,5.045000
